# Execution Researcher API

Offline execution research on the Nautilus engine (decision note DEC-017): `backtest_execution` (BacktestEngine), `urgency_analysis` (Component 6 - Trade Scheduling), `slippage_report` (weekly review, feedback 7->1).

In [ ]:
import pandas as pd
from quant_api.core import DataConfig, load_bars
from quant_api.execution import (
    ExecutionConfig, backtest_execution, urgency_analysis, slippage_report,
    execution_algorithms,
)

print(execution_algorithms.describe())

In [ ]:
# Urgency: is it act-now or slice?
print(urgency_analysis(2, half_spread_bps=4.0, impact_bps=1.0,
                       alpha_decay_per_bar=10.0, value_of_1bp=1000.0))
print(urgency_analysis(2, half_spread_bps=40.0, impact_bps=60.0,
                       alpha_decay_per_bar=1.0, value_of_1bp=1000.0))

In [ ]:
# Offline execution backtest on a 1-day catalog window.
WINDOW = DataConfig(start="2026-08-28", end="2026-08-29")
df = load_bars(WINDOW)
targets = df["ts"].iloc[[60, 120, 180]].reset_index(drop=True).to_frame(name="ts")
targets["target_contracts"] = [2, -1, 0]
report = backtest_execution(targets, config=ExecutionConfig(cooldown_secs=0.0, min_gap_contracts=0),
                            data=WINDOW, algo="marketable_limit")
print({k: report[k] for k in ("n_orders", "n_fills", "n_rejected", "slippage_bps_mean", "algo")})
print(report["fills"][:3])

In [ ]:
# Weekly slippage review (synthetic fills; save=False in a notebook).
fills = [{"price": 100.10, "decision_mid": 100.00, "side": 1, "ts": "2026-08-28T09:05:00"},
         {"price": 99.90, "decision_mid": 100.00, "side": -1, "ts": "2026-08-28T09:15:00"}]
print(slippage_report(fills=fills, save=False))